# AI4EAC Liquidity Stress Prediction: End-to-End Champion Pipeline

This notebook executes the complete 5-stage tournament reproduction and ensemble stacking pipeline.

### Pipeline Architecture:
1. **Stage 1**: Tournament Anchor Reproduction (Multi-Seed GBDT + Stepwise Hill Climbing on 87 features).
2. **Stage 2**: 4-Seed Domain GBDT Zoo (CatBoost, XGBoost, LightGBM with domain-specific feature slices).
3. **Stage 3**: Dual TabPFN Foundation Priors (Physics & Solvency ratio representations).
4. **Stage 4**: Diversity Base Learners (MultiStrata 5-Fold Stratified Ensemble + 10-Fold Distillation Students + Neural TabMLP).
5. **Stage 5**: 10-Fold Cross-Validated Logit-Space L2 Regularized Meta-Stacker + Forward Hill Climbing + Natural Prevalence Calibration.
6. **Stage 6**: Automated Competition Invariant Audit (30k rows, probability clipping, zero NaNs, baseline alignment).

In [1]:
# Cell 0: Environment Setup, Sys.Path Resolution, and GPU Inspection
!pip install -q feature-engine tabpfn
import sys, os, time
from pathlib import Path
import torch

# Ensure repository root is in python path
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Working Directory: {repo_root}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("submissions", exist_ok=True)

Working Directory: /kaggle/working
CUDA Available: True
GPU Device: Tesla T4
Device Count: 2


## Stage 1: Tournament Baseline Anchor Reproduction & Hill Climbing
Trains 4-seed CatBoost, XGBoost, and HistGradientBoosting on the 87 selected features, optimizes weights via SLSQP, applies Beta/Platt calibration, and runs stepwise hill climbing.

In [1]:
from src.stages.stage1_reproduce import run_stage1

t0 = time.time()
s1_score = run_stage1()
print(f"✓ Stage 1 Complete in {(time.time() - t0)/60:.2f} min | Composite Score: {s1_score:.5f}")

NameError: name 'time' is not defined

## Stage 2: 4-Seed Multi-Model Domain GBDT Zoo
Executes CatBoost, XGBoost, and LightGBM across 4 random seeds on specialized domain subsets (solvency ratios, balance collapses, and triage columns).

In [ ]:
from src.stages.stage2_gbdt_zoo import run_stage2

t0 = time.time()
s2_score = run_stage2(seeds=(42, 100, 2024, 777), n_splits=5)
print(f"✓ Stage 2 Complete in {(time.time() - t0)/60:.2f} min | Composite Score: {s2_score:.5f}")

## Stage 3: Dual TabPFN Foundation Priors
Extracts non-tree foundation model representations on GPU using RankGauss Quantile Transformation on the physical solvency ratio views.

In [ ]:
from src.stages.stage3_tabpfn_priors import run_stage3

t0 = time.time()
s3_score = run_stage3(n_splits=10)
print(f"✓ Stage 3 Complete in {(time.time() - t0)/60:.2f} min | Composite Score: {s3_score:.5f}")

## Stage 4: Diversity Pipeline (MultiStrata, Distillation Students, Neural TabMLP)
Executes:
1. MultiStrata 5-Fold Iterative Stratification Ensemble (906 features).
2. 10-Fold Self-Training Distillation Students (CatBoost & XGBoost) on 70k soft pseudo-targets.
3. PyTorch Neural TabMLP with residual connections and bounded logit scaling.

In [ ]:
from src.stages.stage4_diversity import run_stage4

t0 = time.time()
s4_score = run_stage4()
print(f"✓ Stage 4 Complete in {(time.time() - t0)/60:.2f} min | Composite Score: {s4_score:.5f}")

## Stage 5: 10-Fold Cross-Validated Logit-Space L2 Meta-Stacker
Solves analytical L-BFGS-B regularized regression in logit space across all 13 model streams, performs forward-selection hill-climbing, applies asymmetric tail smoothing, and enforces strict natural prevalence alignment to 0.15340.

In [ ]:
from src.stages.stage5_meta_stacker import run_stage5

t0 = time.time()
final_champion_score = run_stage5()
print(f"★ STAGE 5 META-STACKER COMPLETE: Champion Score = {final_champion_score:.5f} ★")

## Stage 6: Final Verification & Invariants Audit
Checks all 5 hard competition invariants: exact row counts (30,000), column names, zero NaNs, probability bounding [0.002, 0.998], and benchmark correlation.

In [ ]:
import pandas as pd
from src.stages.audit import run_audit

# Run automated audit
audit_passed = run_audit()
print(f"Audit Passed: {audit_passed}")

# Inspect final submission preview
sub_file = "submissions/submission_stage5_final.csv"
if os.path.exists(sub_file):
    df_sub = pd.read_csv(sub_file)
    print(f"Submission Shape: {df_sub.shape}")
    print(f"Mean Predicted Probability: {df_sub['Target'].mean():.5f}")
    print(f"5th Percentile: {df_sub['Target'].quantile(0.05):.5f} | 95th Percentile: {df_sub['Target'].quantile(0.95):.5f}")
    print("First 10 Rows:")
    print(df_sub.head(10))
else:
    print(f"{sub_file} not found yet.")